# Proyecto

## Introducción

## Objetivo

## Hipótesis
El **cambio en la cartera de crédito** mexicana **no es sensible** a cambios **reputacionales** en el sector financiero. En cambio **sí es sensible** a cambios en las variables **macroeconómicas.**

## Fuentes de Información

## Unidad Muestral
**Porcentaje de desgaste o sobrevivencia** de saldo de crédito por tipo de crédito de **mes a mes.**

# Desarrollo de la TAD

## Librerías

In [13]:
%time
# sistema
import os

# manipulacion datos
import pandas as pd
import numpy as np

# graficación
import matplotlib.pyplot as plt
import plotly.express as px
import seaborn as sns

CPU times: user 2 μs, sys: 1 μs, total: 3 μs
Wall time: 4.05 μs


## Importar Datos
Aunque de manera inicial se había utilizado un función generada con Gemini, se opta por usar una función más sencilla propia. De esta manera poder visualizar de manera mas controlada el contenido de cada carpeta y poder segmentar cada origen de datos y su tratamiento previo a la unión de la tablas.

In [14]:
# ruta con archivos
ruta_bases = "/home/brauliocb126/Documents/Diplomado/Proyecto/Bases" #cpu
#ruta_bases = "/home/brauliocbd15/Documents/Diplomado/Proyecto/Bases" #laptop

### Método Gemini
Este método se escribió con ayuda de Gemini, por lo que se deben tomar consideraciones.

In [15]:
def camina_carpetas(ruta_busqueda):
    frames_ = {}

    # Recorrer de forma recursiva subcarpetas y archivos con os.walk
    for raiz, carpetas, archivos in os.walk(ruta_busqueda):
        for archivo in archivos:
            # Obtener la ruta completa del archivo
            ruta_completa = os.path.join(raiz, archivo)
            
            # Extraer el nombre sin extensión y la extensión en minúsculas
            nombre_sin_ext, extension = os.path.splitext(archivo)
            extension = extension.lower()

            try:
                # Identificar el formato por la extensión
                if extension in ['.csv', '.txt']:
                    frames_[nombre_sin_ext] = pd.read_csv(ruta_completa)
                elif extension in ['.parquet', '.pq']:
                    frames_[nombre_sin_ext] = pd.read_parquet(ruta_completa)
                elif extension in ['.xlsx', '.xls']:
                    frames_[nombre_sin_ext] = pd.read_excel(ruta_completa)
                elif extension == '.json':
                    frames_[nombre_sin_ext] = pd.read_json(ruta_completa)
                elif extension in ['.feather', '.ft']:
                    frames_[nombre_sin_ext] = pd.read_feather(ruta_completa)
                else:
                    print(f"⚠️ Formato omitido: {archivo}")
                    continue

                print(f"✅ Cargado exitosamente: {os.path.relpath(ruta_completa, ruta_busqueda)}")

            except Exception as e:
                print(f"❌ Error al cargar {archivo}: {e}")

    for nombre_var, f in frames_.items():
        # Limpiamos el nombre para asegurar que sea un nombre de variable válido en Python
        nombre_limpio = nombre_var.replace(" ", "_").replace("-", "_")
        
        # Asignamos el DataFrame a una variable global con ese nombre
        globals()[nombre_limpio] = f
        print(f"Variable creada: {nombre_limpio}")

    return frames_

### Crédito Banxico

In [16]:
# información de créditos vigentes banxico
ruta_cred_banxico = ruta_bases + "/banxico_20260906"

In [17]:
frames_banxico = camina_carpetas(ruta_busqueda = ruta_cred_banxico)

✅ Cargado exitosamente: credito_personal.xlsx
✅ Cargado exitosamente: credito_hipotecario_media_residencial.xlsx
✅ Cargado exitosamente: credito_nomina.xlsx
✅ Cargado exitosamente: credito_consumo_duradero.xlsx
✅ Cargado exitosamente: credito_tarjeta_credito.xlsx
✅ Cargado exitosamente: credito_hipotecario_vivineda_social.xlsx
Variable creada: credito_personal
Variable creada: credito_hipotecario_media_residencial
Variable creada: credito_nomina
Variable creada: credito_consumo_duradero
Variable creada: credito_tarjeta_credito
Variable creada: credito_hipotecario_vivineda_social


/home/brauliocb126/Documents/Diplomado/Proyecto/pyenv_personal/lib/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/home/brauliocb126/Documents/Diplomado/Proyecto/pyenv_personal/lib/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/home/brauliocb126/Documents/Diplomado/Proyecto/pyenv_personal/lib/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
/home/brauliocb126/Documents/Diplomado/Proyecto/pyenv_personal/lib/python3.14/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply op

In [18]:
nom = list(frames_banxico.keys())
nom

['credito_personal',
 'credito_hipotecario_media_residencial',
 'credito_nomina',
 'credito_consumo_duradero',
 'credito_tarjeta_credito',
 'credito_hipotecario_vivineda_social']

In [19]:
for i in nom:
    n = i.split(sep="_")[1]
    columnas_credito = {'Banco de México': 'fecha',
                        'Unnamed: 1': 'saldo_vigente_'+n}
    mod =  globals()[i]
    mod.drop(index=np.arange(0,17,1), inplace=True)
    mod.rename(columns=columnas_credito, inplace=True)
    mod['fecha'] = pd.to_datetime(mod['fecha'])
    mod['saldo_vigente_'+n] = mod['saldo_vigente_'+n].astype('float64')
    display(i, mod)

'credito_personal'

,fecha,saldo_vigente_personal
17,2011-02-01,42880.6
18,2011-03-01,45666.3
19,2011-04-01,48004.1
20,2011-05-01,50076.6
21,2011-06-01,51562.1
...,...,...
198,2026-03-01,284344.0
199,2026-04-01,286438.6
200,2026-05-01,287516.5
201,2026-06-01,290675.2


'credito_hipotecario_media_residencial'

,fecha,saldo_vigente_hipotecario
17,2003-12-01,102171.2
18,2004-01-01,102803.6
19,2004-02-01,103189.7
20,2004-03-01,103895.1
21,2004-04-01,104989.5
...,...,...
284,2026-03-01,1443593.2
285,2026-04-01,1465497.0
286,2026-05-01,1453447.4
287,2026-06-01,1478693.2


'credito_nomina'

...

,fecha,saldo_vigente_consumo
17,2003-12-01,19151.4
18,2004-01-01,19975.3
19,2004-02-01,20626.8
20,2004-03-01,22400.9
21,2004-04-01,23346.9
...,...,...
284,2026-03-01,89320.7
285,2026-04-01,87534.4
286,2026-05-01,86751.1
287,2026-06-01,89672.3


'credito_tarjeta_credito'

,fecha,saldo_vigente_tarjeta
17,2003-12-01,57924.8
18,2004-01-01,57952.1
19,2004-02-01,58940.4
20,2004-03-01,61107.5
21,2004-04-01,63730.4
...,...,...
284,2026-03-01,671719.8
285,2026-04-01,681828.6
286,2026-05-01,686256.5
287,2026-06-01,693620.0


'credito_hipotecario_vivineda_social'

,fecha,saldo_vigente_hipotecario
17,2003-12-01,19234.3
18,2004-01-01,19032.1
19,2004-02-01,19308.5
20,2004-03-01,19146.5
21,2004-04-01,19001.5
...,...,...
284,2026-03-01,57798.9
285,2026-04-01,40372.8
286,2026-05-01,59937.9
287,2026-06-01,40330.8


### CNSF

In [25]:
# información de comision de seguros y fianzas
ruta_cnsf = ruta_bases + "/cnsf_20260904"

In [26]:
frames_cnsf = camina_carpetas(ruta_busqueda= ruta_cnsf)

✅ Cargado exitosamente: info_estadistica_credito_siniestros.csv
✅ Cargado exitosamente: info_estadistica_credito_vivienda_siniestros.csv
✅ Cargado exitosamente: info_sanciones_aseguradoras.csv
✅ Cargado exitosamente: info_participacion_mercado_seguros.csv
✅ Cargado exitosamente: info_estadistica_credito_emision.csv
✅ Cargado exitosamente: info_estadistica_credito_vivienda_clientes.csv
✅ Cargado exitosamente: info_seguros_estado_resultados_sector_consolidado.csv
✅ Cargado exitosamente: info_estadistica_credito_vivienda_datos_credito.csv
✅ Cargado exitosamente: info_seguros_operaciones_ramos_practicados_sector_asegurador.csv
✅ Cargado exitosamente: info_estadistica_credito_vivienda_cobranza.csv
Variable creada: info_estadistica_credito_siniestros
Variable creada: info_estadistica_credito_vivienda_siniestros
Variable creada: info_sanciones_aseguradoras
Variable creada: info_participacion_mercado_seguros
Variable creada: info_estadistica_credito_emision
Variable creada: info_estadistica_cr

In [37]:
nom_cnsf = list(frames_cnsf.keys())
nom_cnsf

['info_estadistica_credito_siniestros',
 'info_estadistica_credito_vivienda_siniestros',
 'info_sanciones_aseguradoras',
 'info_participacion_mercado_seguros',
 'info_estadistica_credito_emision',
 'info_estadistica_credito_vivienda_clientes',
 'info_seguros_estado_resultados_sector_consolidado',
 'info_estadistica_credito_vivienda_datos_credito',
 'info_seguros_operaciones_ramos_practicados_sector_asegurador',
 'info_estadistica_credito_vivienda_cobranza']

In [43]:
for i in nom_cnsf:
    display(i, globals()[i])

'info_estadistica_credito_siniestros'

,anio,plazo,tipo_insolvencia,entidad,pais,tipo_credito,giro,tipo_poliza,moneda,forma_venta,numero_siniestros,monto_siniestro,monto_pagado,monto_deducible,monto_coaseguros,gastos_ajuste,salvamentos,monto_recuperacion,recuperacion_reaseguro
0,2015,1,Mora prolongada,Puebla,No aplica (interno),Interno,Fabricacion de productos de plastico,Global,Extranjera,Agentes Persona Fisica,1,0,0,0,0,7460,0,NaN,NaN
1,2015,5,Mora prolongada,Distrito Federal,No aplica (interno),Interno,Otras industrias alimentarias,Global,Nacional,Fuerza de Venta Interna o Casa Matriz,1,-24242,0,0,0,3124,0,NaN,NaN
2,2015,7,Insolvencia de hecho,Estado de Mexico,No aplica (interno),Interno,"Comercio al por menor de combustibles, aceites...",Global,Nacional,Agentes Persona Fisica,2,0,0,0,0,157241,425880,NaN,NaN
3,2015,7,Mora prolongada,Distrito Federal,No aplica (interno),Interno,Otras industrias alimentarias,Global,Nacional,Fuerza de Venta Interna o Casa Matriz,1,-32612,0,0,0,0,0,NaN,NaN
4,2015,7,Mora prolongada,Sonora,No aplica (interno),Interno,Otro producto al Comercio al por mayor,Global,Nacional,Directo,1,0,0,0,0,522,0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21982,2025,365,Mora prolongada,Ciudad de Mexico,No aplica (interno),Interno,"Comercio al por mayor de maquinaria, mobiliari...",Global,Extranjera,Agentes Persona Fisica,2,874419,97474,51136,9747,1514,0,0.0,874419.0
21983,2025,365,Mora prolongada,Ciudad de Mexico,No aplica (interno),Interno,"Edicion de periodicos, revistas, libros y simi...",Global,Nacional,Agentes Persona Fisica,1,0,0,0,0,52023,0,0.0,0.0
21984,2025,366,Mora prolongada,Yucatan,No aplica (interno),Mixto,Fabricacion de maquinaria y equipo para las ac...,Global,Nacional,Agentes Persona Moral,1,0,0,0,0,825,0,0.0,0.0
21985,2025,470,Mora prolongada,Puebla,No aplica (interno),Mixto,Servicios relacionados con la mineria,Global,Extranjera,Agentes Persona Fisica,1,0,43426,0,18611,0,0,0.0,0.0


'info_estadistica_credito_vivienda_siniestros'

,anio,causa_incumplimiento,evento_origen_siniestro,numero_creditos,monto_siniestro,monto_recuperado_reaseguro,monto_pagado,valor_ultimo_avaluo,valor_recuperado
0,2015,Habitante diferente al acreditado / coacreditado,Periodo establecido en la poliza desde el inic...,1,69131,0,69131,637000,0
1,2015,Divorcio o separacion o viudez,Adjudicacion al Intermediario Financiero,79,10702193,56157,5883128,40888684,0
2,2015,Negacion de pago,Dacion en Pago (cliente entrega titulo y poses...,2,362115,0,362115,2192000,0
3,2015,Habitante diferente al acreditado / coacreditado,Adjudicacion a un tercero,7,743147,0,569408,2708970,0
4,2015,Negacion de pago,Adjudicacion a un tercero,23,2722715,17727,1107110,12023463,0
...,...,...,...,...,...,...,...,...,...
487,2025,Cambio de residencia,Periodo establecido en la poliza desde el inic...,1,321336,0,321336,2442000,0
488,2025,Insolvencia,Periodo establecido en la poliza desde el inic...,23,11986072,0,10684187,89117000,0
489,2025,Otra causa de incumplimiento,Periodo establecido en la poliza desde el inic...,8,3819364,0,3819364,22796000,0
490,2025,Preferencia por el pago hacia otras deudas,Liquidación del crédito asegurado con descuento,1,481690,0,481690,4694000,0


'info_sanciones_aseguradoras'

...

,anio,clave_administrador,apoyo_financiamiento,tipo_tasa,tipo_cartera,moneda,plazo_credito,porcentaje_cobertura,frecuencia_pago,numero_creditos,monto_enganche,monto_credito,prima_emitida,prima_cedida,prima_devengada,saldo_principal
0,2015,ABC Capital,Infonavit Apoyo Infonavit,Tasa Fija,Interes Social,UDI,300,25.0,Mensual,5,205481,1302609,8296,0,8296,1207248
1,2015,ABC Capital,Infonavit Apoyo Infonavit,Tasa Fija,Interes Social,UDI,300,30.0,Mensual,4,56747,1078199,8152,0,8152,802724
2,2015,ABC Capital,Infonavit Apoyo Infonavit,Tasa Fija,Media y Residencial,UDI,240,30.0,Mensual,6,214188,3325112,24655,0,24655,2749162
3,2015,ABC Capital,Infonavit Apoyo Infonavit,Tasa Fija,Media y Residencial,UDI,300,15.0,Mensual,1,122820,477180,2148,0,2148,429228
4,2015,ABC Capital,Infonavit Cofinavit,Tasa Fija,Interes Social,UDI,240,10.0,Mensual,2,141455,394419,1260,0,1260,226307
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
84547,2025,Adamantine Servicios,Sin Apoyo,Tasa Fija,Interes Social,UDI,300,25.0,Mensual,53,1856082,12512466,58562,0,58562,7909599
84548,2025,Adamantine Servicios,Sin Apoyo,Tasa Fija,Interes Social,UDI,360,25.0,Mensual,134,4279724,29115936,315096,0,315096,36016482
84549,2025,Adamantine Servicios,Sin Apoyo,Tasa Fija,Media y Residencial,UDI,300,15.0,Mensual,1,175000,575000,4199,0,4199,613757
84550,2025,Adamantine Servicios,Sin Apoyo,Tasa Fija,Media y Residencial,UDI,300,25.0,Mensual,10,1195307,4995834,23515,0,23515,3099449


'info_seguros_operaciones_ramos_practicados_sector_asegurador'

,FECHA,INSTITUCION,VIDA,PENSIONES,ACC_ENF,SALUD,DANOS_SA,AUTOS,GARANTIA,CRED
0,31/12/2019,"Armour Secure Insurance, S.A.de C.V.",no,no,no,no,si,no,no,no
1,31/12/2019,"A.N.A. Compañía de Seguros, S.A. de C.V.",no,no,no,no,no,si,no,no
2,31/12/2019,"Agroasemex, S.A.",si,no,si,no,si,si,no,no
3,31/12/2019,"AIG Seguros México, S.A. de C.V.",no,no,si,no,si,si,no,no
4,31/12/2019,"Allianz México, S.A., Compañía de Seguros",si,no,si,no,si,si,no,no
...,...,...,...,...,...,...,...,...,...,...
714,31/12/2025,"Virginia Surety Seguros de México, S.A. de C.V.",no,no,no,no,si,si,no,no
715,31/12/2025,"Zurich Aseguradora Mexicana, S.A. de C.V.",si,no,si,no,si,si,no,no
716,31/12/2025,"Zurich Santander Seguros México, S.A.",si,no,si,no,si,si,no,no
717,31/12/2025,"Zurich Vida, Compañía de Seguros, S.A.",si,no,si,no,no,no,no,no


'info_estadistica_credito_vivienda_cobranza'

,anio,numero_mensualidades_no_pagadas,numero_meses_mora,numero_creditos,saldo_inicial_principal
0,2015,0,0,213121,110000000000
1,2015,3,1,3,902498
2,2015,6,1,2,1124063
3,2015,22,1,1,361281
4,2015,25,1,1,246900
...,...,...,...,...,...
6944,2025,24,3,3,8004278
6945,2025,48,12,14,9792752
6946,2025,20,4,1,2669850
6947,2025,44,7,1,183014


### Condusef

In [ ]:
# información de comision de defensa a usuarios de entidades financieras
ruta_condusef = ruta_bases + "/condusef_20260906"

In [28]:
frames_condusef = camina_carpetas(ruta_busqueda= ruta_condusef)

✅ Cargado exitosamente: tabla_calificaciones_producto_financiero_institucion_sector.xlsx
✅ Cargado exitosamente: portal_buro_entidades_financieras.xlsx
Variable creada: tabla_calificaciones_producto_financiero_institucion_sector
Variable creada: portal_buro_entidades_financieras


In [39]:
nom_condusef = list(frames_condusef.keys())
nom_condusef

['tabla_calificaciones_producto_financiero_institucion_sector',
 'portal_buro_entidades_financieras']

In [42]:
for i in nom_condusef:
    display(i, globals()[i])

'tabla_calificaciones_producto_financiero_institucion_sector'

,CALIFICACIONES DE SUPERVISIÓN,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6
0,Periodo: Enero - junio 2024 (Base de datos del...,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Sector,Institución Financiera,Producto,RECA,EvaInformacionProducto,EvaSupervisionProducto,CalificacionPromedioProducto
3,Banca Múltiple,"Banco Nacional de México, S. A., Integrante de...",Crédito en Cuenta Corriente,0300-440-026067/06-03695-0921,NaN,10,10
4,Banca Múltiple,"Banco Nacional de México, S. A., Integrante de...",Pagaré (PRLV),0300-003-000667/27-03262-0821,NaN,9.4,9.4
...,...,...,...,...,...,...,...
627,NaN,NaN,NaN,NaN,NaN,NaN,NaN
628,NaN,NaN,NaN,NaN,NaN,NaN,NaN
629,NaN,NaN,NaN,NaN,NaN,NaN,NaN
630,SOFOM ER,NaN,NaN,NaN,NaN,NaN,NaN


'portal_buro_entidades_financieras'

,SECTOR,Id sector,Id,INSTITUCION,TOTAL DE RECLAMACIONES CONDUSEF Y REUNE,INDICE DE RECLAMACION,RECLAMACIONES CONDUSEF,PORCENTAJE DE RESOLUCION FAVORABLE,TIEMPO DE RESPUESTA EN DIAS,RECLAMACIONES REUNE,...,INDICADOR DE RENDIMIENTO SB0,INDICADOR DE RENDIMIENTO SB 55-59,INDICADOR DE RENDIMIENTO SB 60-64,INDICADOR DE RENDIMIENTO SB 65-69,INDICADOR DE RENDIMIENTO SB 70-74,INDICADOR DE RENDIMIENTO SB 75-79,INDICADOR DE RENDIMIENTO SB 80-84,INDICADOR DE RENDIMIENTO SB 85-89,INDICADOR DE RENDIMIENTO SB 90-94,INDICADOR DE RENDIMIENTO SB 1000
0,Administradoras de fondos para el retiro y emp...,4,4,"Afore XXI Banorte, S.A. de C.V.",1113,4.407149,1113,41.100917,11.015357,-,...,4.8408,4.9316,5.2095,5.3883,5.5514,5.6402,5.8976,5.8935,5.7806,5.6356
1,Administradoras de fondos para el retiro y emp...,4,7,"Profuturo AFORE, S.A. de C.V.",483,3.488297,474,29.059829,15.667368,9,...,4.3653,4.9274,5.7441,5.6807,6.0985,6.236,6.3434,6.4462,6.2298,6.1737
2,Administradoras de fondos para el retiro y emp...,4,8,"Principal Afore, S.A. de C.V., Principal Grupo...",404,4.978859,404,30.287206,20.483544,-,...,4.9139,5.192,5.3883,5.4334,5.5226,5.6856,5.3778,5.4841,5.3626,5.2237
3,Administradoras de fondos para el retiro y emp...,4,10,"Afore Sura, S.A. de C.V.",601,4.205998,587,38.602941,18.506352,14,...,5.0158,5.1819,5.7964,5.6618,5.8225,5.8677,5.9986,5.9868,5.9303,5.9397
4,Administradoras de fondos para el retiro y emp...,4,12,"Afore Inbursa, S.A. de C.V., Grupo Financiero ...",126,4.560641,110,35.849057,10.06422,16,...,5.241,5.7975,5.4941,5.6668,5.797,6.0057,6.1043,6.1812,6.2626,6.173
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3586,ENTIDADES QUE DEJARON DE SER SOFOM,No Aplica,No Aplica,"Tiempo de Creer, S.A.P.I. de C.V., SOFOM, E.N.R.",No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,...,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica
3587,ENTIDADES QUE DEJARON DE SER SOFOM,No Aplica,No Aplica,"Smart Bird, S.A. de C.V., SOFOM, E.N.R.",No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,...,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica
3588,ENTIDADES QUE DEJARON DE SER SOFOM,No Aplica,No Aplica,"Jumafri, S.A. de C.V., SOFOM, E.N.R.",No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,...,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica
3589,ENTIDADES QUE DEJARON DE SER SOFOM,No Aplica,No Aplica,"CT Soluciones México, S.A. de C.V., SOFOM, E.N.R.",No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,...,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica,No Aplica


### Macro Económicas

### INEGI
No se omite, pero no pienso hacerlo ahorita